# Task 2: Baseline and improved ticket classifier
**Project:** Support Ticket Classification for VoltMart Electronics

What this notebook does:
1. Loads `cleaned_tickets.csv` (made in Task 1) and separates the **text** and **label** columns.
2. Builds a **baseline** model: TF-IDF + Logistic Regression, scored with 5-fold cross-validation.
3. Builds an **improved** model and shows that its mean F1 is higher than the baseline's.
4. Runs a **leakage check**: no identical ticket text appears in both the train and test part of any fold.
5. Saves the best model as `best_model.joblib` and checks that it loads again.

Run it from top to bottom with **Kernel -> Restart & Run All**.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
RANDOM_STATE = 42

## 1. Load the data and separate text and label
* **text** = the ticket message (this is what the model reads, called `X`)
* **label** = the category such as `email` or `website` (this is what the model must guess, called `y`)

In [2]:
df = pd.read_csv("cleaned_tickets.csv")
print("Rows, columns:", df.shape)
df.head()

Rows, columns: (2839, 2)


,text,label
0,e-poçtuma daxil olmaqda çətinlik çəkirəm. hər dəfə daxil olmaq istəyəndə 'ya...,email
1,veb saydım ishlemir ve men onu berpa ede bilmirem. men serveri yeniden bashl...,website
2,komandamın bir üzvü üçün yeni e-poçt hesabı qurmaqda köməyə ehtiyacım var. t...,email
3,veb saytım çox yavaş yüklənir və bunun səbəbini bilmirəm. bəzi performans te...,website
4,men coxlu spam mekdublar alıram ve onları neje dayandırajaghımı bilmirem. me...,email


In [3]:
# Same ticket text twice would let the model "see the answer" in training. Remove repeats first.
before = len(df)
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset="text").reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} after dropping repeated ticket texts")

X = df["text"]
y = df["label"]
print()
print("Tickets per category:")
print(y.value_counts())

Rows: 2839 -> 2839 after dropping repeated ticket texts

Tickets per category:
label
email      1594
website    1245
Name: count, dtype: int64


## 2. Cross-validation setup
**5-fold cross-validation** splits the data into 5 parts. The model is trained on 4 parts and tested on the
remaining part. This is repeated 5 times so every ticket is used for testing once. We report the **mean** score.

* **Accuracy** = share of tickets given the right category.
* **F1 (macro)** = balance of precision and recall, averaged over all categories, so small categories count as much as big ones.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "f1_macro": "f1_macro"}

def evaluate(name, pipeline):
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)
    return {
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),
    }

## 3. Baseline: TF-IDF + Logistic Regression

In [5]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
baseline_result = evaluate("Baseline: TF-IDF + LogisticRegression", baseline)
pd.DataFrame([baseline_result]).round(4)

,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,Baseline: TF-IDF + LogisticRegression,0.9824,0.005,0.9821,0.0051


## 4. Improved models
Two simple ideas:
* **Word n-grams (1-2 words)** so the model sees short phrases such as "yeni hesab", not only single words.
* **LinearSVC**, a linear model that often works better than Logistic Regression for text.

The ticket text is Azerbaijani, which has many word endings, so I also try **character n-grams**
(pieces of 2-5 letters inside words). They cope better with different word endings.

In [6]:
candidates = {
    "Word 1-2 grams + LogisticRegression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Word 1-2 grams + LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", LinearSVC(random_state=RANDOM_STATE)),
    ]),
    "Char 2-5 grams + LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
        ("clf", LinearSVC(random_state=RANDOM_STATE)),
    ]),
}

candidate_results = [evaluate(name, pipe) for name, pipe in candidates.items()]
pd.DataFrame(candidate_results).round(4)

,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,Word 1-2 grams + LogisticRegression,0.9817,0.0044,0.9813,0.0045
1,Word 1-2 grams + LinearSVC,0.9905,0.0024,0.9903,0.0024
2,Char 2-5 grams + LinearSVC,0.9968,0.0017,0.9968,0.0018


## 5. Comparison table: baseline vs improved

In [7]:
# The improved model = the candidate with the highest mean F1.
best_candidate = max(candidate_results, key=lambda r: r["f1_macro_mean"])
improved_name = best_candidate["model"]
improved_pipeline = candidates[improved_name]

comparison = pd.DataFrame([baseline_result, best_candidate])
comparison.insert(0, "role", ["Baseline", "Improved"])
print(comparison.round(4).to_string(index=False))

gain = best_candidate["f1_macro_mean"] - baseline_result["f1_macro_mean"]
print()
print(f"Improved F1 - Baseline F1 = {gain:+.4f}")
if gain > 0:
    print("RESULT: the improved model has a HIGHER mean F1 than the baseline.")
else:
    print("WARNING: the improved model is NOT higher than the baseline. Try other settings.")

    role                                 model  accuracy_mean  accuracy_std  f1_macro_mean  f1_macro_std
Baseline Baseline: TF-IDF + LogisticRegression         0.9824        0.0050         0.9821        0.0051
Improved            Char 2-5 grams + LinearSVC         0.9968        0.0017         0.9968        0.0018

Improved F1 - Baseline F1 = +0.0147
RESULT: the improved model has a HIGHER mean F1 than the baseline.


## 6. Leakage check
**Data leakage** happens when the same ticket is in both the training part and the test part, which makes the
score look better than it really is. For every fold we check that no ticket text appears in both parts.

In [8]:
leaks = 0
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    overlap = set(X.iloc[train_idx]) & set(X.iloc[test_idx])
    leaks += len(overlap)
    print(f"Fold {fold}: train={len(train_idx)}, test={len(test_idx)}, identical tickets in both = {len(overlap)}")

assert leaks == 0, "Leakage found: identical tickets appear in train and test folds"

# NO DATA LEAKAGE: repeated ticket texts were removed before splitting, and the check above
# confirms that no identical ticket appears in both the train and test part of any fold.
print()
print("No data leakage: 0 identical tickets shared between train and test in all 5 folds.")

Fold 1: train=2271, test=568, identical tickets in both = 0
Fold 2: train=2271, test=568, identical tickets in both = 0
Fold 3: train=2271, test=568, identical tickets in both = 0
Fold 4: train=2271, test=568, identical tickets in both = 0
Fold 5: train=2272, test=567, identical tickets in both = 0

No data leakage: 0 identical tickets shared between train and test in all 5 folds.


## 7. Save the best model
We pick the pipeline with the higher mean F1 and **fit it on all the data**.
The saved file contains the whole pipeline (TF-IDF and classifier), so later it can take raw ticket text directly.

In [9]:
if best_candidate["f1_macro_mean"] > baseline_result["f1_macro_mean"]:
    best_name, best_pipeline = improved_name, improved_pipeline
else:
    best_name, best_pipeline = baseline_result["model"], baseline

best_pipeline.fit(X, y)
joblib.dump(best_pipeline, "best_model.joblib")
print("Saved best_model.joblib ->", best_name)

Saved best_model.joblib -> Char 2-5 grams + LinearSVC


## 8. Check that the saved model loads and predicts

In [10]:
loaded = joblib.load("best_model.joblib")
samples = X.sample(5, random_state=RANDOM_STATE).tolist()
for text, pred in zip(samples, loaded.predict(samples)):
    print(f"{pred:>10}  <-  {text[:90]}")
print()
print("best_model.joblib loaded and predicted without error.")

   website  <-  veb saytımda yeni ssl sertifikatı quraşdırmağa çalışıram, lakin səhv mesajı alıram. siz kö
     email  <-  komandam üçün yeni e-poçt hesabı qurmaqda köməyə ehtiyacım var. biz şirkət domenindən e-po
     email  <-  telefonumdakı outlook proqramından istifadə edərək e-poçt hesabıma daxil ola bilmirəm.
   website  <-  veb sayt yavaş yüklənir və mən orada naviqasiya etməkdə çətinlik çəkirəm. hər hansı bir tə
     email  <-  yeni e-poçt hesabı yaratmağa çalışıram, lakin səhv mesajı alıram. nə baş verdiyini anlamağ

best_model.joblib loaded and predicted without error.
